# BigLunge 2D mask audit (pipeline-faithful)

**Purpose:** visually verify that the algorithm-generated BigLunge tumor
masks are correctly localized on the slices the training pipeline
actually feeds the model.

**Key correctness constraint:** the cached `slice_idx` values are in
the RAS + `Spacing((1,1,2))` grid, NOT the native NIfTI grid. An
earlier version of this notebook loaded the NIfTI directly with
`nib.as_closest_canonical` (reorient only, no Z resampling) and then
indexed with the cached `slice_idx`, which put us at a completely
wrong anatomical depth — the mask looked absent even though the
pipeline had no bug. This version reuses the real pipeline
transforms (`LoadImaged → Orientationd(RAS) → Spacingd(1,1,2) →
SliceSelectd`) so what we visualize is exactly what the model sees.

**What to eyeball:**
1. Does the red mask contour sit on a plausible soft-tissue density
   inside the lung parenchyma?
2. Does the green 96×96 crop box cover the mask?
3. Does the resulting crop show a lesion-like density or random
   chest content?

If >20% of audited rows fail (1) or (2), the algorithmic mask is the
real bottleneck for BigLunge fine-tune and we should reconsider the
tumor-centered crop.

In [ ]:
import os
import random
import sys
import warnings
from collections import Counter
from pathlib import Path
from typing import Dict, List

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings('ignore')

REPO = Path('/home/hansstem/SCLC-Classification')
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from data.dataset_2d import get_biglunge_2d_data_list  # noqa: E402
from data.transforms import SliceSelectd  # noqa: E402
from monai.transforms import (  # noqa: E402
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
)

DATA_PATH = '/home/data/TrainingData'
CSV_PATH = '/home/data/TrainingData/patients_parameters.csv'
CACHE_ROOT = os.path.expanduser('~/.cache/monai_biglunge_2d/img224')
PATCH_SIZE = 96                       # matches CropAroundTumord
PIXDIM = (1.0, 1.0, 2.0)              # matches Spacingd in _build_2d_pipeline
MASK_SUFFIX = '_label_tc.nii.gz'

N_PATIENTS = 18                       # 6 per class if possible
SLICES_PER_PATIENT = 2                # first-quartile + median tumor slice
SEED = 42

OUT_DIR = REPO / 'data_exploration' / 'output' / 'biglunge_expl'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = {0: 'Adenocarcinoma', 1: 'Small Cell', 2: 'Squamous'}
random.seed(SEED); np.random.seed(SEED)
print(f'Output dir: {OUT_DIR}')

## 1. Load the data list the training pipeline builds

Reuses `get_biglunge_2d_data_list` so every `(patient, slice_idx)`
pair we audit is one the fine-tune trained on. `max_slices_per_volume=None`
here so we can audit the full tumor extent — training uses a cap of 8.

In [ ]:
splits = get_biglunge_2d_data_list(
    data_path=DATA_PATH,
    csv_path=CSV_PATH,
    cache_root=CACHE_ROOT,
    tumor_mask_suffix=MASK_SUFFIX,
    val_frac=0.15, test_frac=0.15, seed=SEED,
    max_slices_per_volume=None,
)

per_patient: Dict[str, Dict] = {}
for split_name, entries in splits.items():
    for e in entries:
        pid = str(e['patient_id'])
        rec = per_patient.setdefault(pid, {
            'patient_id': pid,
            'image': e['image'],
            'tumor_mask': e['tumor_mask'],
            'scan_label': int(e['scan_label']),
            'split': split_name,
            'slices': [],
        })
        rec['slices'].append(int(e['slice_idx']))
for rec in per_patient.values():
    rec['slices'] = sorted(set(rec['slices']))

print(f'Total patients with non-empty masks: {len(per_patient)}')
print('Class distribution:', Counter(r["scan_label"] for r in per_patient.values()))
print('Split distribution:', Counter(r["split"] for r in per_patient.values()))

## 2. Tumor-slice coverage per patient

In [ ]:
slice_counts = np.array([len(r['slices']) for r in per_patient.values()])
print(f'Tumor slices per patient: n={len(slice_counts)}, '
      f'min={slice_counts.min()}, p25={np.percentile(slice_counts, 25):.0f}, '
      f'median={int(np.median(slice_counts))}, p75={np.percentile(slice_counts, 75):.0f}, '
      f'max={slice_counts.max()}')
tiny = [r for r in per_patient.values() if 0 < len(r['slices']) <= 2]
print(f'Patients with 1-2 tumor slices (suspicious): {len(tiny)}')
for r in tiny[:6]:
    print(f'  {r["patient_id"]} ({CLASS_NAMES[r["scan_label"]]}): slices={r["slices"]}')

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.hist(slice_counts, bins=40, color='steelblue', edgecolor='black')
ax.set_xlabel('Tumor slices per patient (after Orient+Spacing, >= 1 px)')
ax.set_ylabel('Patients')
ax.set_title('BigLunge algorithmic mask — slice coverage per patient')
ax.axvline(np.median(slice_counts), color='red', linestyle='--', label=f'median={int(np.median(slice_counts))}')
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'slice_count_hist.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Pipeline-faithful per-sample loader

Runs the same deterministic transform prefix the training pipeline
uses, up to and including `SliceSelectd`. This gives us the
`(image, mask)` pair on the exact slice the model consumes, in the
2 mm-resampled RAS grid. Everything downstream (crop box, crop
visualization) is computed on that pair, so nothing can drift
between what we draw and what the pipeline trains on.

In [ ]:
prefix = Compose([
    LoadImaged(keys=['image', 'tumor_mask']),
    EnsureChannelFirstd(keys=['image', 'tumor_mask'], channel_dim='no_channel'),
    Orientationd(keys=['image', 'tumor_mask'], axcodes='RAS'),
    Spacingd(
        keys=['image', 'tumor_mask'], pixdim=PIXDIM,
        mode=['bilinear', 'nearest'],
    ),
    SliceSelectd(keys=['image', 'tumor_mask'], slice_key='slice_idx'),
])


def load_pipeline_slice(entry: Dict) -> Dict:
    """Return dict with 2D numpy arrays `image` and `mask` for the
    single selected slice, in the pipeline's grid."""
    sample = prefix({**entry})
    img = sample['image']
    mask = sample['tumor_mask']
    arr_img = np.asarray(img.cpu() if hasattr(img, 'cpu') else img)
    arr_mask = np.asarray(mask.cpu() if hasattr(mask, 'cpu') else mask)
    # Expected shape: (1, H, W, 1) — drop channel + Z
    img2d = arr_img[0, ..., 0]
    mask2d = (arr_mask[0, ..., 0] > 0.5).astype(np.uint8)
    return {
        'image': img2d,
        'mask': mask2d,
        'volume_shape_after_spacing': tuple(arr_img.shape),
    }


def centroid_and_box(mask_slice: np.ndarray, patch: int = PATCH_SIZE):
    idx = np.argwhere(mask_slice > 0)
    if idx.size == 0:
        cx, cy = mask_slice.shape[0] // 2, mask_slice.shape[1] // 2
    else:
        cx, cy = idx.mean(axis=0).round().astype(int).tolist()
    x0 = cx - patch // 2
    y0 = cy - patch // 2
    return (cx, cy), (x0, y0, patch, patch)


def crop_with_padding(arr: np.ndarray, x0: int, y0: int, patch: int):
    X, Y = arr.shape
    x1 = x0 + patch; y1 = y0 + patch
    xs0, xs1 = max(0, x0), min(X, x1)
    ys0, ys1 = max(0, y0), min(Y, y1)
    core = arr[xs0:xs1, ys0:ys1]
    out = np.zeros((patch, patch), dtype=arr.dtype)
    out[xs0 - x0: xs0 - x0 + core.shape[0], ys0 - y0: ys0 - y0 + core.shape[1]] = core
    return out


def window_hu(img: np.ndarray, lo: float = -1024, hi: float = 400) -> np.ndarray:
    """Display windowing. Pipeline uses ScaleIntensityRanged(-1024, 3071, 0, 1)
    which compresses soft-tissue contrast at display time — this lighter
    window is just for human-eye readability, it does not affect the model.
    """
    a = np.clip(img, lo, hi)
    return (a - lo) / (hi - lo)

## 4. Stratified random sample across classes

In [ ]:
by_class: Dict[int, List[Dict]] = {0: [], 1: [], 2: []}
for r in per_patient.values():
    by_class[r['scan_label']].append(r)

per_class = max(1, N_PATIENTS // 3)
selected: List[Dict] = []
rng = random.Random(SEED)
for c in (0, 1, 2):
    pool = by_class[c]
    rng.shuffle(pool)
    selected.extend(pool[:per_class])
for r in selected:
    print(f'  {r["patient_id"]} | {CLASS_NAMES[r["scan_label"]]:<15} | split={r["split"]:<5} | #slices={len(r["slices"])}')

## 5. Per-sample audit plots

One row per (patient, slice) pair. Columns:
1. Slice the model sees + red mask contour.
2. Same slice + green 96×96 crop box + centroid marker.
3. The 96×96 crop — this is what `_build_2d_pipeline` hands the CNN
   (before resize to 224×224 and intensity normalization).

In [ ]:
rows_data: List[Dict] = []
for rec in selected:
    slices_sorted = rec['slices']
    if not slices_sorted:
        continue
    picks_idx = sorted(set([
        slices_sorted[max(0, len(slices_sorted) // 4)],
        slices_sorted[len(slices_sorted) // 2],
    ]))[:SLICES_PER_PATIENT]
    for z in picks_idx:
        entry = {
            'image': rec['image'],
            'tumor_mask': rec['tumor_mask'],
            'slice_idx': int(z),
        }
        try:
            pair = load_pipeline_slice(entry)
        except Exception as e:
            print(f'[skip] {rec["patient_id"]} z={z}: {e}')
            continue
        rows_data.append({
            'patient_id': rec['patient_id'],
            'label': rec['scan_label'],
            'slice_idx': z,
            **pair,
        })

n_rows = len(rows_data)
print(f'Prepared {n_rows} rows for plotting.')
fig, axes = plt.subplots(n_rows, 3, figsize=(13, 4.2 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, 3)

for i, row in enumerate(rows_data):
    img2d = row['image']
    mask2d = row['mask']
    (cx, cy), (x0, y0, w, h) = centroid_and_box(mask2d)
    crop_img = crop_with_padding(img2d, x0, y0, PATCH_SIZE)
    wide = window_hu(img2d)

    ax = axes[i, 0]
    ax.imshow(wide.T, cmap='gray', origin='lower')
    if mask2d.sum() > 0:
        ax.contour(mask2d.T, levels=[0.5], colors='red', linewidths=1.0, origin='lower')
    ax.set_title(
        f'{row["patient_id"]} | {CLASS_NAMES[row["label"]]} | z={row["slice_idx"]} '
        f'(post-Spacing) | mask px={int(mask2d.sum())}',
        fontsize=9,
    )
    ax.axis('off')

    ax = axes[i, 1]
    ax.imshow(wide.T, cmap='gray', origin='lower')
    ax.add_patch(mpatches.Rectangle(
        (x0, y0), w, h, linewidth=1.2, edgecolor='lime', facecolor='none',
    ))
    ax.plot(cx, cy, marker='+', color='lime', markersize=10, mew=1.5)
    ax.set_title(f'crop box (96x96), centroid=({cx},{cy})', fontsize=9)
    ax.axis('off')

    ax = axes[i, 2]
    ax.imshow(window_hu(crop_img).T, cmap='gray', origin='lower')
    ax.set_title('what the model sees (96x96, pre-resize)', fontsize=9)
    ax.axis('off')

plt.tight_layout()
grid_path = OUT_DIR / 'mask_audit_grid.png'
plt.savefig(grid_path, dpi=110, bbox_inches='tight')
plt.show()
print(f'Saved: {grid_path}')